# End-to-End Subtitle Generator and Summarizer Pipeline

This notebook demonstrates the complete progression from a raw educational `.mp4` video to generated `.srt` subtitle files and concise `.txt` summaries using OpenAI Whisper and Hugging Face BART.

**File Structure Expectations:**
* `videos/lecture1.mp4` must exist before running Audio Extraction.
* Outputs populate dynamically into `audio/`, `transcripts/`, `subtitles/`, `summaries/`, `reports/`.

*Helper functions `chunk_transcript` and `convert_to_srt` are imported from `utils.py` — no logic is duplicated here.*


## 1. Setup and Imports

In [ ]:
import os, sys, json, time
from pathlib import Path

# Add project root to sys.path so utils.py is importable
project_root = Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import whisper
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM  # direct instantiation — pipeline("summarization") is broken in this transformers build
import ffmpeg
import jiwer
from rouge_score import rouge_scorer
from IPython.display import display, Markdown

# Shared custom pipeline utilities (no duplicate logic)
from utils import chunk_transcript, convert_to_srt

VIDEO_NAME = "lecture1"  # Demo target — change to lecture2 or lecture3 as needed
PATHS = {
    "video":          project_root / "videos"            / f"{VIDEO_NAME}.mp4",
    "audio":          project_root / "audio"             / f"{VIDEO_NAME}.wav",
    "transcript_txt": project_root / "transcripts"       / f"{VIDEO_NAME}.txt",
    "transcript_json":project_root / "transcripts"       / f"{VIDEO_NAME}_segments.json",  # matches transcribe.py output
    "srt":            project_root / "subtitles"         / f"{VIDEO_NAME}.srt",
    "summary":        project_root / "summaries"         / f"{VIDEO_NAME}_summary.txt",
    "ref_transcript": project_root / "references"        / f"{VIDEO_NAME}.txt",
    "ref_summary":    project_root / "reference_summaries" / f"{VIDEO_NAME}_summary.txt",
}

# Ensure output directories exist
for key in ("audio","transcript_txt","transcript_json","srt","summary"):
    PATHS[key].parent.mkdir(parents=True, exist_ok=True)

print(f"✅ Setup complete. Demo target: '{VIDEO_NAME}'")


## 2. Audio Extraction
Extracts **mono 16 kHz WAV** audio using `ffmpeg-python` — the format Whisper requires.


In [ ]:
def extract_audio(vid_path: Path, aud_path: Path):
    if not vid_path.exists():
        print(f"⚠️  {vid_path.name} not found — skipping extraction.")
        return
    print(f"Extracting audio from {vid_path.name}...")
    try:
        (
            ffmpeg.input(str(vid_path))
            .output(str(aud_path), ac=1, ar=16000, format="wav")
            .overwrite_output()
            .run(quiet=True)
        )
        size_mb = aud_path.stat().st_size / 1e6
        print(f"✅ Saved {aud_path.name} ({size_mb:.1f} MB)")
    except ffmpeg.Error as e:
        print(f"❌ ffmpeg error: {e}")

extract_audio(PATHS["video"], PATHS["audio"])


## 3. Whisper Transcription (with Segments)
Loads Whisper `base` (GPU if available). Saves both a plain `.txt` transcript and a
segment-level `.json` file containing `start`, `end`, and `text` fields per block.


In [ ]:
def run_transcription(audio_path, txt_path, json_path):
    if not audio_path.exists():
        print("⚠️  Audio file missing — skipping transcription.")
        return
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading Whisper 'base' on {device}...")
    model = whisper.load_model("base", device=device)
    print(f"Transcribing {audio_path.name}...  (may take a few minutes)")
    start = time.time()
    result = model.transcribe(str(audio_path))  # returns {'text': ..., 'segments': [...]}
    elapsed = time.time() - start
    print(f"✅ Done in {elapsed:.1f}s — {len(result['text'].split())} words")
    txt_path.write_text(result["text"].strip(), encoding="utf-8")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({"segments": result["segments"]}, f, indent=2)
    print(f"Saved: {txt_path.name}, {json_path.name}")

run_transcription(PATHS["audio"], PATHS["transcript_txt"], PATHS["transcript_json"])


## 4. SRT Subtitle Generation
`convert_to_srt()` from `utils.py` converts Whisper segments into the official SRT format:
- Sequential index numbers
- `HH:MM:SS,mmm --> HH:MM:SS,mmm` timestamps
- Subtitle text block
- Blank line separator


In [ ]:
def generate_srt(json_path, srt_path):
    if not json_path.exists():
        print("⚠️  Segments JSON missing — skipping SRT generation.")
        return
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    segments = data.get("segments", [])
    result_path = convert_to_srt(segments, srt_path)  # calls utils.convert_to_srt
    blocks = [b for b in srt_path.read_text().split("\n\n") if b.strip()]
    print(f"✅ {srt_path.name} written — {len(blocks)} subtitle blocks")

generate_srt(PATHS["transcript_json"], PATHS["srt"])


## 5. Transcript Chunking
`chunk_transcript()` from `utils.py` splits the raw transcript into ~200-word chunks
at **sentence boundaries** (via `nltk.sent_tokenize`) so BART never receives a token
sequence exceeding its 1024-token input limit.


In [ ]:
chunks = []
if PATHS["transcript_txt"].exists():
    raw_text = PATHS["transcript_txt"].read_text(encoding="utf-8")
    chunks = chunk_transcript(raw_text, max_words=200)  # calls utils.chunk_transcript
    print(f"✅ {len(chunks)} chunks created")
    print(f"   Chunk 1 preview ({len(chunks[0].split())} words): {chunks[0][:120]}...")
else:
    print("⚠️  Transcript missing — run transcription cell first.")


## 6. Summarization with BART
Uses `facebook/bart-large-cnn` via direct `AutoTokenizer` / `AutoModelForSeq2SeqLM`
instantiation (the high-level `pipeline("summarization")` is unavailable in this
`transformers` build). Each chunk is summarized individually. If the combined result
exceeds 100 words, a second inference pass enforces the ≤100-word constraint.

**Fallback**: swap `model_name` for `"google/flan-t5-base"` on CPU-only machines.


In [ ]:
def run_summarization(chunks, summary_path):
    if not chunks:
        print("⚠️  No chunks — run chunking cell first.")
        return
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = "facebook/bart-large-cnn"
    print(f"Loading {model_name} on {device}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

    intermediate = []
    for idx, chunk in enumerate(chunks, 1):
        print(f"  Summarizing chunk {idx}/{len(chunks)}...")
        inputs = tokenizer(chunk, max_length=1024, return_tensors="pt", truncation=True).to(device)
        ids = model.generate(inputs["input_ids"], max_length=60, min_length=15, num_beams=4, early_stopping=True)
        intermediate.append(tokenizer.decode(ids[0], skip_special_tokens=True))

    combined = " ".join(intermediate)
    print(f"Combined: {len(combined.split())} words")

    # Enforce ≤100-word hard limit
    if len(combined.split()) > 100:
        print("Enforcing ≤100 word constraint (pass 2)...")
        inputs = tokenizer(combined, max_length=1024, return_tensors="pt", truncation=True).to(device)
        ids = model.generate(inputs["input_ids"], max_length=95, min_length=30, num_beams=4, early_stopping=True)
        combined = tokenizer.decode(ids[0], skip_special_tokens=True)

    summary_path.write_text(combined.strip(), encoding="utf-8")
    print(f"✅ Summary saved ({len(combined.split())} words)")

run_summarization(chunks, PATHS["summary"])


## 7. WER Evaluation
Computes Word Error Rate using `jiwer`. Requires a reference transcript in
`references/lecture1.txt`. If absent, instructions for creating one are shown.


In [ ]:
if PATHS["ref_transcript"].exists() and PATHS["transcript_txt"].exists():
    ref = " ".join(PATHS["ref_transcript"].read_text(encoding="utf-8").split()).lower()
    hyp = " ".join(PATHS["transcript_txt"].read_text(encoding="utf-8").split()).lower()
    wer = jiwer.wer(ref, hyp)
    print(f"Word Error Rate (WER): {wer:.2%}")
    print("✅ PASS" if wer < 0.30 else "⚠️  FAIL — consider Whisper 'medium' or 'large'")
else:
    print("⚠️  Reference transcript not found.")
    print(f"  → Create '{PATHS['ref_transcript']}' by manually typing the first 1-2 minutes of the audio.")


## 8. ROUGE Evaluation
Computes ROUGE-1, ROUGE-2, and ROUGE-L F1 scores using `rouge-score`. Requires a
2–3 sentence human reference summary in `reference_summaries/lecture1_summary.txt`.


In [ ]:
if PATHS["ref_summary"].exists() and PATHS["summary"].exists():
    ref_sum = PATHS["ref_summary"].read_text(encoding="utf-8").strip()
    hyp_sum = PATHS["summary"].read_text(encoding="utf-8").strip()
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    scores = scorer.score(ref_sum, hyp_sum)
    print("ROUGE F1 Scores:")
    print(f"  ROUGE-1: {scores['rouge1'].fmeasure:.4f}")
    print(f"  ROUGE-2: {scores['rouge2'].fmeasure:.4f}")
    print(f"  ROUGE-L: {scores['rougeL'].fmeasure:.4f}")
else:
    print("⚠️  Reference summary not found.")
    print(f"  → Write a 2–3 sentence summary and save it to '{PATHS['ref_summary']}'")


## 9. Results Display
Renders the generated `.srt` file and final summary inline using `IPython.display`.


In [ ]:
print("=" * 50)
print("         PIPELINE COMPLETE — RESULTS")
print("=" * 50 + "\n")

# Display SRT preview
if PATHS["srt"].exists():
    srt_preview = "\n".join(PATHS["srt"].read_text(encoding="utf-8").splitlines()[:24])
    display(Markdown(f"### Generated Subtitles (`{PATHS['srt'].name}` — first 6 blocks)\n```\n{srt_preview}\n```"))
else:
    print("⚠️  SRT file not found.")

# Display summary inline
if PATHS["summary"].exists():
    summary_txt = PATHS["summary"].read_text(encoding="utf-8").strip()
    display(Markdown(f"### Final Summary (`{PATHS['summary'].name}` — {len(summary_txt.split())} words)\n> {summary_txt}"))
else:
    print("⚠️  Summary file not found.")
